# 07 – Verodostojnost odgovora (faithfulness)

Da li je svaka tvrdnja u odgovoru podržana **delovima dokumenta koje je generator dobio** (kontekst) – za razliku od tačnosti (06_judge), koja odgovor poredi sa očekivanim odgovorom.

- Ocenjivač razloži odgovor na tvrdnje i za svaku kaže `supported` / `unsupported` / `contradicted`; metrika se računa u `src/faithfulness.py`.
- Meri se izabrana konfiguracija (Gemma 4 12B + prompt v2b); Qwen 7B v2b je opcioni kontrast.
- Rezultat: `faithfulness_<TAG>.jsonl` u folderu pokretanja, kopija na Drive i zip.

Runtime: **GPU (T4)**.

## 1. Repo i paketi

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/minjaaaa/Schneider-Electric-Internship.git repo
%cd repo
!git log -1 --format="%h %ci %s"
!pip install -q -U transformers
!pip install -q python-dotenv bitsandbytes accelerate

## 2. Hugging Face token

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 3. Podešavanja

In [ ]:
JUDGE_MODEL = "Qwen/Qwen2.5-14B-Instruct"     # druga porodica od Gemme (izabrani generator)
MULTIMODAL = False
FAITH_VERSION = "f1"
TAG = "qwen14b_f1"                           # ime fajla: faithfulness_<TAG>.jsonl
MAX_NEW_TOKENS = 600                         # lista tvrdnji je duža od verdikta tačnosti
RUNS = [
    "2026-10-18_1716_gen_prompt-v2b_gemma4-12b",      # izabrana konfiguracija
    # "2026-10-17_1837_gen_prompt-v2b_qwen7b",        # opcioni kontrast (~15 min)
]
assert FAITH_VERSION in TAG, "TAG mora da sadrži FAITH_VERSION (npr. qwen14b_f1)"

## 4. Kontekst (isti delovi dokumenta koje je video generator) i model ocenjivača

In [ ]:
import sys, json
sys.path.insert(0, "src")
from chunking import load_jsonl
from config import DATA_PROCESSED, GEN_RESULTS_DIR
from figures import figure_chunks
from generate import HFChatModel, format_context
from faithfulness import faithfulness_run, faithfulness_answer, faithfulness_fingerprint

by_id = {c["chunk_id"]: c for c in load_jsonl(DATA_PROCESSED / "chunks_section.jsonl") + figure_chunks()}

def contexts_for(run):
    """{id pitanja: tekst izvora sa oznakama} - iz context_ids sačuvanih uz svaki odgovor."""
    rows = [json.loads(l) for l in open(GEN_RESULTS_DIR / run / "answers.jsonl", encoding="utf-8")]
    return {r["id"]: format_context([by_id[cid] for cid in r["context_ids"]], "labeled") for r in rows}

contexts = {run: contexts_for(run) for run in RUNS}
print(len(by_id), "delova dokumenta;", {r: len(c) for r, c in contexts.items()}, "| prompt:",
      FAITH_VERSION, faithfulness_fingerprint(FAITH_VERSION))

llm = HFChatModel(JUDGE_MODEL, max_new_tokens=MAX_NEW_TOKENS, multimodal=MULTIMODAL)

## 5. Proba na jednom odgovoru
Qwen 7B v2b, Wi-Fi za Lisabon (odgovor „NS-Guest-LIS“ je izmišljen) – očekuje se `unsupported`. Pokreni pre punog ocenjivanja.

In [ ]:
probe_run = "2026-10-17_1837_gen_prompt-v2b_qwen7b"
row = next(json.loads(l) for l in open(GEN_RESULTS_DIR / probe_run / "answers.jsonl", encoding="utf-8")
           if json.loads(l)["id"] == "Q14")
r = faithfulness_answer(row, contexts_for(probe_run)["Q14"], llm, FAITH_VERSION)
print("ODGOVOR:", row["answer"])
print("OCENJIVAČ:", r["judge_raw"])
print("tvrdnje:", [(c["claim"], c["verdict"]) for c in r["claims"]], "| verodostojnost:", r.get("faithfulness"),
      "| greška:", r["judge_error"], "|", r["judge_latency_s"], "s")

## 6. Google Drive (kopija posle svakog pokretanja)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/northstar_results/faithfulness"
os.makedirs(DRIVE_DIR, exist_ok=True)

## 7. Ocenjivanje

In [ ]:
import shutil
from pathlib import Path
from faithfulness import summarize

json.dump({"judge_model": JUDGE_MODEL, "faithfulness_version": FAITH_VERSION,
           "faithfulness_hash": faithfulness_fingerprint(FAITH_VERSION), "context_style": "labeled",
           "max_new_tokens": MAX_NEW_TOKENS, "runs": [Path(r).name for r in RUNS]},
          open(f"{DRIVE_DIR}/faithfulness_config_{TAG}.json", "w"), indent=2)

for run in RUNS:
    name = Path(run).name
    out = GEN_RESULTS_DIR / name / f"faithfulness_{TAG}.jsonl"
    if out.exists() and sum(1 for _ in open(out, encoding="utf-8")) == len(contexts[name]):
        print(name, "- već ocenjeno, preskačem")
    else:
        out = faithfulness_run(GEN_RESULTS_DIR / name, contexts[name], llm, TAG, FAITH_VERSION)
    shutil.copy(out, f"{DRIVE_DIR}/{name}__{out.name}")
    s = summarize([json.loads(l) for l in open(out, encoding="utf-8")])
    print(name, "| tvrdnji:", s["n_claims"], s["claims"], "| prosečna verodostojnost:", s["mean_faithfulness"],
          "| potpuno utemeljeni:", s["fully_grounded_answers"], "| greške:", len(s["judge_errors"]))

## 8. Preuzimanje
Ako ova ćelija padne (npr. dozvola za Drive), fajlovi su već na Drive-u u `northstar_results/faithfulness`.

In [ ]:
!cd {DRIVE_DIR} && zip -q /content/faithfulness_{TAG}.zip *{TAG}*
from google.colab import files
files.download(f"/content/faithfulness_{TAG}.zip")